# Experiments 6 — Second-LLM Backbone Robustness

Goal: change only the representation-generation LLM backbone while keeping the frozen descriptions, part IDs, and shared connections from Experiments 2 and 5.

Primary comparison: Anchor vs Global vs Hierarchy Local on (1) static shared-edge realization / OBB overlap and (2) scale-perturbation margin drift.

PartNet uses 11 deterministic primary-complete objects per category (121 total). OpenWorld uses all Experiments-5 primary complete cases (expected 119).


In [1]:
import os
import importlib
import pandas as pd
import methods
from IPython.display import display

importlib.reload(methods)
from methods import *


## Cell 2 — Configure the frozen primary result directories and the second backbone

Default second backbone: Qwen3.7-Plus through an OpenAI-compatible endpoint, non-thinking mode.
Set `SECOND_LLM_BASE_URL` to your Model Studio workspace endpoint, or edit `e6.secondary_base_url` directly.


In [2]:
e6 = Experiments6Config()
e6.project_dir = "."
e6.output_dir = r"experiments6_results\qwen3_7_plus"
e6.primary_partnet_dir = r"partnet_results\experiments2"
e6.primary_openworld_dir = r"openworld_results"
e6.secondary_label = "qwen3.7-plus"
e6.secondary_model = "qwen3.7-plus"
e6.secondary_base_url = "https://maas.qianwenaiapi.com/compatible-mode/v1"
e6.secondary_api_key_env = '''sk-ws-H.PPIPXHL.v7xV.MEUCIAHrO3UsiMeVOtF4a28BpY27WF32q4y1WiKkdTZLmFz6AiEAuoWeP5FZIVTx1XM8E94tWmyNTsfWkJfTtoM6AT2DIac
'''
e6.secondary_extra_body = {"enable_thinking": False}
e6.temperature = 0.0
e6.max_tokens = 8192
e6.max_attempts = 3
e6.workers = 4
e6.timeout = 180.0
e6.transport_retries = 2
e6.partnet_per_category = 11
e6.subset_seed = 606
e6.scale_factors = (0.8, 1.2, 1.5)
e6.bootstrap_samples = 10000
e6.statistics_seed = 42
e6.allow_api = True

print("secondary model:", e6.secondary_model)
print("PartNet source:", e6.project_path(e6.primary_partnet_dir))
print("OpenWorld source:", e6.project_path(e6.primary_openworld_dir))
print("output:", e6.output_path())


secondary model: qwen3.7-plus
PartNet source: D:\MyJupyter\Works\ASPT\partnet_results\experiments2
OpenWorld source: D:\MyJupyter\Works\ASPT\openworld_results
output: D:\MyJupyter\Works\ASPT\experiments6_results\qwen3_7_plus


## Cell 3 — Freeze the exact evaluation cohort

This reads only existing Experiments-2/5 files. No API call.


In [3]:
manifest = build_experiments6_manifest(e6)

print("PartNet objects:", manifest["selection"]["partnet_count"])
print("OpenWorld objects:", manifest["selection"]["openworld_count"])

display(pd.DataFrame(manifest["partnet_objects"]).groupby("category").size().rename("count").to_frame())
display(pd.DataFrame(manifest["openworld_objects"]).groupby("category").size().rename("count").to_frame())


Experiments 6 manifest: PartNet=121, OpenWorld=119, secondary=qwen3.7-plus
PartNet objects: 121
OpenWorld objects: 119


,count
category,
Bottle,11
Chair,11
Faucet,11
Knife,11
Lamp,11
Laptop,11
Mug,11
Scissors,11
StorageFurniture,11


,count
category,
Appliances,20
Electronics,20
Furniture,20
MechanicalComposite,19
Tools,20
Vehicles,20


## Cell 4 — Load the frozen shared specs and primary DeepSeek results

No GT geometry is sent to the second model. No API call.


In [4]:
sources = prepare_experiments6_sources(e6, manifest)

for dataset in ("partnet", "openworld"):
    print(dataset, "objects:", len(sources[dataset]["inputs"]["objects"]))
    print(dataset, "shared valid:", sum(row["status"] == "valid" for row in sources[dataset]["shared"].values()))


partnet objects: 121
partnet shared valid: 121
openworld objects: 119
openworld shared valid: 119


## Cell 5 — Create the second-backbone client

For current Alibaba Model Studio endpoints, replace the workspace ID in the regional OpenAI-compatible base URL. If you use a different OpenAI-compatible provider, set its base URL/model/key environment variable and set `secondary_extra_body={}` unless that provider supports the Qwen parameter.


In [8]:
if e6.allow_api and not e6.secondary_base_url:
    raise ValueError(
        "Set SECOND_LLM_BASE_URL or edit e6.secondary_base_url. "
        "Example Beijing format: https://<WorkspaceId>.cn-beijing.maas.aliyuncs.com/compatible-mode/v1"
    )

client = None

if e6.allow_api:
    client = create_openai_compatible_client(
        api_key_env=e6.secondary_api_key_env,
        base_url=e6.secondary_base_url,
        timeout=e6.timeout,
    )

print("client ready" if client is not None else "API disabled; cache-only mode")


client ready


## Cell 6 — Generate Anchor / Global / Local with the second backbone

This is the only API-intensive cell. Results are cached per object/method and resume safely.


In [17]:
generations = run_experiments6_generation(
    sources,
    e6,
    client=client,
)

for dataset in ("partnet", "openworld"):
    print()
    print(dataset)
    for method in REPRESENTATIONS:
        rows = generations[dataset][method]
        valid = sum(row.get("status") == "valid" for row in rows.values())
        print(method, "valid:", valid, "/", len(rows))


anchor: valid=121/121
global: 2/2
global: valid=121/121
local: valid=121/121
anchor: 4/4
anchor: valid=117/119
global: 20/106
global: 40/106
global: 60/106
global: 80/106
global: 100/106
global: 106/106
global: valid=119/119
local: 20/119
local: 40/119
local: 60/119
local: 80/119
local: 100/119
local: 119/119
local: valid=119/119

partnet
anchor valid: 121 / 121
global valid: 121 / 121
local valid: 121 / 121

openworld
anchor valid: 117 / 119
global valid: 119 / 119
local valid: 119 / 119


## Cell 7 — Disable all further API use

Everything below is offline.


In [18]:
e6.allow_api = False
print("API disabled for evaluation/statistics")


API disabled for evaluation/statistics


## Cell 8 — Static OBB structural evaluation


In [19]:
evaluations = run_experiments6_evaluation(
    sources,
    generations,
    e6,
)

for dataset in ("partnet", "openworld"):
    print(
        dataset,
        "secondary three-method complete:",
        len(evaluations[dataset]["complete_ids"]),
        "/",
        len(sources[dataset]["inputs"]["objects"]),
    )


Open-world static evaluation: complete=121/121; evaluation failures=0
Open-world static evaluation: complete=117/119; evaluation failures=0
partnet secondary three-method complete: 121 / 121
openworld secondary three-method complete: 117 / 119


## Cell 9 — Scale perturbation robustness

Same factors as the formal PartNet/OpenWorld experiments: 0.8, 1.2, 1.5. No API.


In [20]:
perturbations = run_experiments6_perturbation(
    sources,
    generations,
    evaluations,
    e6,
)

for dataset in ("partnet", "openworld"):
    print(dataset, "perturbation records:", len(perturbations[dataset]["objects"]))


All-edge scale perturbation: 1089 method-object-scale records；未调用 LLM。
All-edge scale perturbation: 1065 method-object-scale records；未调用 LLM。
partnet perturbation records: 1089
openworld perturbation records: 1065


## Cell 10 — Cross-backbone statistics

Primary DeepSeek is re-read from the frozen Experiments-2/5 outputs and restricted to the same objects that are valid for all three second-backbone methods.


In [21]:
results6 = summarize_experiments6_results(
    sources,
    generations,
    evaluations,
    perturbations,
    e6,
)

print("PARTNET STATIC")
display(experiments6_summary_view(results6, "partnet", "static"))

print("OPENWORLD STATIC")
display(experiments6_summary_view(results6, "openworld", "static"))

print("PARTNET PERTURBATION")
display(experiments6_summary_view(results6, "partnet", "scale_perturbation"))

print("OPENWORLD PERTURBATION")
display(experiments6_summary_view(results6, "openworld", "scale_perturbation"))


PARTNET STATIC


metric                   obb_overlap_pair_fraction  \
backbone         method                              
primary_deepseek anchor                   0.113294   
                 global                   0.234553   
                 local                    0.238546   
qwen3.7-plus     anchor                   0.149603   
                 global                   0.262678   
                 local                    0.308562   

metric                   shared_edge_obb_sat_deviation  
backbone         method                                 
primary_deepseek anchor                       0.016544  
                 global                       0.085728  
                 local                        0.071828  
qwen3.7-plus     anchor                       0.015084  
                 global                       0.102384  
                 local                        0.089420

OPENWORLD STATIC


metric                   obb_overlap_pair_fraction  \
backbone         method                              
primary_deepseek anchor                   0.128051   
                 global                   0.188653   
                 local                    0.208131   
qwen3.7-plus     anchor                   0.167984   
                 global                   0.245865   
                 local                    0.264331   

metric                   shared_edge_obb_sat_deviation  
backbone         method                                 
primary_deepseek anchor                       0.016662  
                 global                       0.052095  
                 local                        0.047096  
qwen3.7-plus     anchor                       0.024511  
                 global                       0.059187  
                 local                        0.053669

PARTNET PERTURBATION


metric                         margin_drift
backbone         scale method              
primary_deepseek 0.8   anchor      0.001635
                       global      0.023965
                       local       0.022922
                 1.2   anchor      0.001464
                       global      0.020422
                       local       0.020375
                 1.5   anchor      0.003480
                       global      0.045181
                       local       0.044612
qwen3.7-plus     0.8   anchor      0.001309
                       global      0.022311
                       local       0.024363
                 1.2   anchor      0.000976
                       global      0.019851
                       local       0.020890
                 1.5   anchor      0.002085
                       global      0.044803
                       local       0.044766

OPENWORLD PERTURBATION


metric                         margin_drift
backbone         scale method              
primary_deepseek 0.8   anchor      0.001790
                       global      0.022224
                       local       0.019092
                 1.2   anchor      0.001458
                       global      0.019836
                       local       0.017637
                 1.5   anchor      0.003524
                       global      0.045221
                       local       0.040161
qwen3.7-plus     0.8   anchor      0.002683
                       global      0.019220
                       local       0.017984
                 1.2   anchor      0.002594
                       global      0.017316
                       local       0.016621
                 1.5   anchor      0.006256
                       global      0.038019
                       local       0.036314

## Cell 11 — Inspect the primary paired conclusions and backbone concordance


In [22]:
stats6 = pd.DataFrame(results6["paired_statistics"])

primary_columns = [
    "dataset",
    "backbone",
    "experiment",
    "baseline",
    "metric",
    "scale",
    "n",
    "anchor_mean",
    "baseline_mean",
    "mean_improvement",
    "ci_low",
    "ci_high",
    "holm_p",
    "wins",
    "ties",
    "losses",
]

display(stats6[primary_columns].round(6))

concordance6 = pd.DataFrame(results6["backbone_concordance"])
display(concordance6)

status6 = pd.DataFrame(results6["generation_status"])
display(status6.groupby("dataset")["secondary_complete"].agg(["sum", "count"]))


,dataset,backbone,experiment,baseline,metric,scale,n,anchor_mean,baseline_mean,mean_improvement,ci_low,ci_high,holm_p,wins,ties,losses
0,partnet,primary_deepseek,static,global,shared_edge_obb_sat_deviation,NaN,121,0.016544,0.085728,0.069185,0.057065,0.081295,0.000000,84,23,14
1,partnet,primary_deepseek,static,global,obb_overlap_pair_fraction,NaN,121,0.113294,0.234553,0.121259,0.090689,0.154077,0.000000,64,44,13
2,partnet,primary_deepseek,static,local,shared_edge_obb_sat_deviation,NaN,121,0.016544,0.071828,0.055284,0.043133,0.066915,0.000000,75,31,15
3,partnet,primary_deepseek,static,local,obb_overlap_pair_fraction,NaN,121,0.113294,0.238546,0.125252,0.095424,0.156120,0.000000,66,45,10
4,partnet,qwen3.7-plus,static,global,shared_edge_obb_sat_deviation,NaN,121,0.015084,0.102384,0.087300,0.078487,0.096332,0.000000,108,8,5
5,partnet,qwen3.7-plus,static,global,obb_overlap_pair_fraction,NaN,121,0.149603,0.262678,0.113075,0.082631,0.144648,0.000000,69,34,18
6,partnet,qwen3.7-plus,static,local,shared_edge_obb_sat_deviation,NaN,121,0.015084,0.089420,0.074337,0.064549,0.083947,0.000000,99,18,4
7,partnet,qwen3.7-plus,static,local,obb_overlap_pair_fraction,NaN,121,0.149603,0.308562,0.158959,0.127594,0.191845,0.000000,77,33,11
8,partnet,primary_deepseek,scale_perturbation,global,margin_drift,0.8,121,0.001635,0.023965,0.022330,0.019936,0.024750,0.000000,117,0,4
9,partnet,primary_deepseek,scale_perturbation,local,margin_drift,0.8,121,0.001635,0.022922,0.021287,0.019078,0.023505,0.000000,118,0,3


,dataset,experiment,baseline,metric,scale,primary_improvement,secondary_improvement,same_direction,secondary_ci_excludes_zero_positive
0,partnet,static,global,shared_edge_obb_sat_deviation,NaN,0.069185,0.087300,True,True
1,partnet,static,local,shared_edge_obb_sat_deviation,NaN,0.055284,0.074337,True,True
2,partnet,static,global,obb_overlap_pair_fraction,NaN,0.121259,0.113075,True,True
3,partnet,static,local,obb_overlap_pair_fraction,NaN,0.125252,0.158959,True,True
4,partnet,scale_perturbation,global,margin_drift,0.8,0.022330,0.021001,True,True
5,partnet,scale_perturbation,local,margin_drift,0.8,0.021287,0.023053,True,True
6,partnet,scale_perturbation,global,margin_drift,1.2,0.018958,0.018875,True,True
7,partnet,scale_perturbation,local,margin_drift,1.2,0.018911,0.019915,True,True
8,partnet,scale_perturbation,global,margin_drift,1.5,0.041701,0.042718,True,True
9,partnet,scale_perturbation,local,margin_drift,1.5,0.041132,0.042682,True,True


,sum,count
dataset,,
openworld,117,119
partnet,121,121


## Cell 12 — Export the review package

Upload only `experiments6_review.zip` for result review.


In [23]:
review_path = export_experiments6_review(e6)
print("upload:", review_path.resolve())


upload: D:\MyJupyter\Works\ASPT\experiments6_results\qwen3_7_plus\experiments6_review.zip
